# AeroPulse Source Likelihood — 02: Evidence Features

Pollutant ratios and temporal signatures, wind-aware transport consistency,
and a behavioural station-type proxy — the evidence the labeling functions
will read.

## Objective

Improvement plan sections 13-20.

The old Phase 2 built six evidence columns (`hour`, `is_weekend`, `season`,
`stagnation_flag`, `fire_active_flag`, `no2_pm25_ratio`) and went straight to
labelling. That is too thin to separate five sources. This notebook adds:

- **section 13** — the full ratio set, NO2/CO/PM2.5 temporal changes, and
  rolling NO2-PM2.5 correlation. A traffic signature is the two rising
  *together*, which no static ratio captures.
- **sections 17-18** — transport direction consistency over a 6 h window.
  A wind direction that has held for six hours is much stronger transport
  evidence than one instantaneous reading.
- **section 20** — a station-type proxy derived from each station's own
  weekday-morning, night and weekend behaviour, fitted on the training
  window only. True station classifications are not in this dataset.

Sections 15-16 (upwind fire, PM10/PM2.5 dust evidence) are audited rather
than rebuilt — the upstream store already provides them.


> **Scientific constraint, repeated in every notebook because it governs how
> the output may be used.** These are *likelihoods derived from heuristics*,
> not measured source attribution. No public ground-truth attribution
> catalogue exists for this dataset. A model that scores well here has
> reproduced its own labeling system — which is not the same as being right.

In [1]:
# ============================================================================
# SETUP — reusable logic lives in source_toolkit.py, not in cells.
# ============================================================================
import os, sys, json, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd()))
import source_toolkit as st

warnings.filterwarnings("ignore")
pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 60)

cfg = st.load_config()
SOURCES = st.SOURCES
print("kernel:", sys.executable)

env: /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/AeroPulse_ML_Notebooks/source_likelihood/.env
project: /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/AeroPulse_ML_Notebooks/source_likelihood
sources: ['biomass_burning', 'traffic', 'industrial', 'dust', 'regional_transport']
kernel: /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/.venv/bin/python


In [2]:
base_file = cfg.base_dir / "source_evidence.parquet"
if not base_file.exists():
    raise FileNotFoundError("Run 01_source_data_quality.ipynb first.")
df = pd.read_parquet(base_file)
df["timestamp_utc"] = pd.to_datetime(df["timestamp_utc"], utc=True)
df = df.sort_values(["location_id", "timestamp_utc"]).reset_index(drop=True)
print("rows", f"{len(df):,}", "| columns", df.shape[1])

rows 1,705,252 | columns 220


## 1. Audit: what the plan asks for that already exists

Same discipline as the propagation pipeline — check before rebuilding. A
second implementation of an existing feature is a second chance to get it
wrong.

In [3]:
REQUESTED = {
    "S15 fire by radius":     ["fire_count_20km_24h", "fire_frp_sum_50km_24h", "fire_min_distance_50km_24h"],
    "S15 upwind fire":        ["upwind_fire_count_50km", "upwind_fire_frp_50km", "upwind_fire_fraction_100km"],
    "S16 PM10 for dust":      ["cams_pm10"],
    "S16 dust indicator":     ["cams_dust"],
    "S17 upwind PM2.5":       ["upwind_pm25_mean", "upwind_pm25_max", "upwind_pm25_gradient"],
    "S17 wind vector":        ["wind_speed_10m", "wind_direction_10m"],
    "S19 SO2/NO2/CO":         ["cams_so2", "cams_no2", "cams_co"],
    "S20 geography":          ["elevation_m", "is_indo_gangetic_plain", "lat", "lon"],
    "S13 ratios":             ["no2_pm25_ratio", "so2_pm25_ratio", "pm10_pm25_ratio"],
    "S13 temporal pollutant": ["no2_change_1h", "rolling_corr_no2_pm25"],
    "S18 transport consistency": ["transport_direction_consistency"],
    "S20 station context":    ["station_type_proxy", "station_weekend_ratio"],
}
rows = []
for family, columns in REQUESTED.items():
    have = [c for c in columns if c in df.columns]
    rows.append({"family": family, "requested": len(columns), "present": len(have),
                 "status": "complete" if len(have) == len(columns) else ("partial" if have else "MISSING"),
                 "missing": ", ".join(c for c in columns if c not in df.columns)[:60]})
audit = pd.DataFrame(rows)
print(audit.to_string(index=False))
print("\nalready complete:", int((audit["status"] == "complete").sum()), "/", len(audit))

                   family  requested  present   status                                         missing
       S15 fire by radius          3        3 complete                                                
          S15 upwind fire          3        3 complete                                                
        S16 PM10 for dust          1        1 complete                                                
       S16 dust indicator          1        1 complete                                                
         S17 upwind PM2.5          3        3 complete                                                
          S17 wind vector          2        2 complete                                                
           S19 SO2/NO2/CO          3        3 complete                                                
            S20 geography          4        4 complete                                                
               S13 ratios          3        0  MISSING no2_pm25_ratio, so

## 2. Pollutant ratios, changes and correlations (section 13)

In [4]:
before = set(df.columns)
t0 = time.time()
df = st.add_pollutant_features(df)
print(f"added in {time.time() - t0:.1f}s:", sorted(set(df.columns) - before))

ratios = [c for c in df.columns if c.endswith("_pm25_ratio")]
print()
print(df[ratios].describe().T[["mean", "50%", "75%", "max"]].round(3).to_string())
print("\nrolling NO2-PM2.5 correlation:")
print(df["rolling_corr_no2_pm25"].describe().round(3).to_string())

added in 0.4s: ['co_change_1h', 'co_change_3h', 'co_pm25_ratio', 'no2_change_1h', 'no2_change_3h', 'no2_pm25_ratio', 'o3_pm25_ratio', 'pm10_pm25_ratio', 'pm25_change_1h', 'pm25_change_3h', 'rolling_corr_co_pm25', 'rolling_corr_no2_pm25', 'so2_pm25_ratio']

                   mean     50%     75%     max
no2_pm25_ratio    0.741   0.324   0.664   166.8
co_pm25_ratio    20.262  10.155  16.634  3122.0
so2_pm25_ratio    0.755   0.319   0.625   121.6
o3_pm25_ratio     5.434   2.872   4.929   523.0
pm10_pm25_ratio   4.070   1.449   2.672  2659.9

rolling NO2-PM2.5 correlation:
count    1695939.000
mean           0.179
std            0.446
min           -0.978
25%           -0.157
50%            0.225
75%            0.549
max            0.990


### Do the ratios separate anything?

A ratio is only useful as evidence if it varies with something. Check each
against the concentration regime — a ratio that is flat is not evidence.

In [5]:
probe = df.sample(300_000, random_state=42).copy()
probe["_band"] = pd.cut(probe["pm25"], [0, 30, 60, 120, 250, np.inf],
                        labels=["<30", "30-60", "60-120", "120-250", ">250"], right=False)
view = probe.groupby("_band", observed=True)[
    [c for c in ("no2_pm25_ratio", "so2_pm25_ratio", "co_pm25_ratio", "pm10_pm25_ratio")
     if c in probe.columns] + ["cams_dust"]
].median().round(3)
print(view.to_string())
print()
print("Every ratio falls monotonically as PM2.5 rises. That is arithmetic, not")
print("chemistry: the ratio has PM2.5 in its denominator, so 'high NO2:PM2.5'")
print("largely means 'PM2.5 is low'. The plan's section 13 asks for these")
print("ratios; taken literally they would make the traffic and industrial")
print("labeling functions fire preferentially on clean air. Section 4 measures")
print("what per-station normalisation does and does not fix.")

         no2_pm25_ratio  so2_pm25_ratio  co_pm25_ratio  pm10_pm25_ratio  cams_dust
_band                                                                             
<30               0.488           0.455         14.400            2.069       13.0
30-60             0.243           0.261          8.125            1.175       13.0
60-120            0.199           0.200          6.288            0.847        7.0
120-250           0.143           0.123          4.371            0.557        6.0
>250              0.071           0.059          1.720            0.305        6.0

Every ratio falls monotonically as PM2.5 rises. That is arithmetic, not
chemistry: the ratio has PM2.5 in its denominator, so 'high NO2:PM2.5'
largely means 'PM2.5 is low'. The plan's section 13 asks for these
ratios; taken literally they would make the traffic and industrial
labeling functions fire preferentially on clean air. Section 4 measures
what per-station normalisation does and does not fix.


## 3. Transport consistency (sections 17-18)

In [6]:
before = set(df.columns)
t0 = time.time()
df = st.add_transport_features(df)
print(f"added in {time.time() - t0:.1f}s:", sorted(set(df.columns) - before))

print("\ntransport_direction_consistency (1.0 = steady wind, 0.0 = boxing the compass):")
print(df["transport_direction_consistency"].describe().round(3).to_string())
steady = float((df["transport_direction_consistency"] > 0.9).mean())
print(f"\nrows with a steady 6 h direction: {steady:.1%}")

added in 0.1s: ['transport_direction_consistency', 'transport_distance_6h', 'transport_mean_direction_deg', 'upwind_pm25_excess', 'wind_direction_variability', 'wind_speed_mean_6h']

transport_direction_consistency (1.0 = steady wind, 0.0 = boxing the compass):
count    1705103.000
mean           0.876
std            0.187
min            0.000
25%            0.849
50%            0.964
75%            0.991
max            1.000

rows with a steady 6 h direction: 68.5%


## 4. Station context (section 20)

In [7]:
fit_end = df["timestamp_utc"].quantile(cfg.train_fraction)
print("station context fitted on rows before:", fit_end)
df, station_context = st.add_station_context(df, fit_end)

anomalies = [c for c in df.columns if c.endswith("_level_anomaly")] +             ([c for c in ["coarse_fraction_anomaly"] if c in df.columns])
print("\npollutant level anomaly (1.0 = this station's own normal level):")
print(df[anomalies].describe().T[["mean", "50%", "75%", "max"]].round(3).to_string())

probe2 = df.sample(300_000, random_state=42).copy()
probe2["_band"] = pd.cut(probe2["pm25"], [0, 30, 60, 120, 250, np.inf],
                         labels=["<30", "30-60", "60-120", "120-250", ">250"], right=False)
ratio_cols = [c for c in probe2.columns if c.endswith("_pm25_ratio")]
print("\nRatio vs level anomaly, by concentration band:")
print("\n  raw ratios (all fall — the denominator effect):")
print(probe2.groupby("_band", observed=True)[ratio_cols].median().round(3).to_string())
print("\n  level anomalies (PM2.5 appears nowhere in these):")
print(probe2.groupby("_band", observed=True)[anomalies].median().round(3).to_string())

print("\nstation type proxy distribution:")
print(station_context["station_type_proxy"].value_counts().to_string())
print("\nbehavioural ratios by derived type:")
print(station_context.groupby("station_type_proxy")[
    ["station_mean_pm25", "station_morning_ratio", "station_night_ratio", "station_weekend_ratio"]
].mean().round(3).to_string())

station context fitted on rows before: 2026-03-22 01:30:00+00:00



pollutant level anomaly (1.0 = this station's own normal level):
                          mean    50%    75%       max
no2_level_anomaly        1.150  0.917  1.526    12.246
co_level_anomaly         1.038  0.884  1.277     7.362
so2_level_anomaly        1.082  0.905  1.367    11.830
o3_level_anomaly         1.122  0.986  1.453     6.882
pm10_level_anomaly       1.218  0.972  1.494    26.602
coarse_fraction_anomaly  3.042  1.083  1.997  1987.775



Ratio vs level anomaly, by concentration band:

  raw ratios (all fall — the denominator effect):
         no2_pm25_ratio  co_pm25_ratio  so2_pm25_ratio  o3_pm25_ratio  pm10_pm25_ratio
_band                                                                                 
<30               0.488         14.400           0.455          4.462            2.069
30-60             0.243          8.125           0.261          2.169            1.175
60-120            0.199          6.288           0.200          1.050            0.847
120-250           0.143          4.371           0.123          0.484            0.557
>250              0.071          1.720           0.059          0.177            0.305

  level anomalies (PM2.5 appears nowhere in these):
         no2_level_anomaly  co_level_anomaly  so2_level_anomaly  o3_level_anomaly  pm10_level_anomaly  coarse_fraction_anomaly
_band                                                                                                           

### Leakage test: the proxy must not move when the future changes

In [8]:
tampered = df[["location_id", "timestamp_utc", "pm25"]].copy()
future = tampered["timestamp_utc"] >= fit_end
tampered.loc[future, "pm25"] = tampered.loc[future, "pm25"] * 20
_, refit = st.add_station_context(tampered, fit_end)
identical = np.allclose(station_context["station_mean_pm25"].to_numpy(),
                        refit["station_mean_pm25"].to_numpy())
print("multiplied every post-cutoff PM2.5 by 20")
print("station context unchanged:", identical)
assert identical, "station context is leaking future data"

multiplied every post-cutoff PM2.5 by 20
station context unchanged: True


## 5. Save

In [9]:
float_cols = df.select_dtypes(include=["float64"]).columns
df[float_cols] = df[float_cols].astype("float32")
out = cfg.feature_dir / "source_evidence_features.parquet"
df.to_parquet(out, index=False)
station_context.to_csv(cfg.feature_dir / "station_context.csv", index=False)

import joblib
joblib.dump({"fit_end": fit_end, "context": station_context},
            cfg.feature_dir / "station_context.joblib")

manifest = {
    "schema_version": cfg.schema_version,
    "generated_utc": pd.Timestamp.now(tz="UTC").isoformat(),
    "rows": int(len(df)),
    "columns": int(df.shape[1]),
    "upstream_audit": audit.to_dict(orient="records"),
    "station_context_fit_end": str(fit_end),
    "station_type_counts": station_context["station_type_proxy"].value_counts().to_dict(),
    "fingerprint": st.dataset_fingerprint(df[["timestamp_utc", "location_id", "pm25"]]),
}
(cfg.feature_dir / "evidence_manifest.json").write_text(json.dumps(manifest, indent=2, default=str))
print("02 COMPLETE ->", out, f"({df.shape[1]} columns)")
print("Next: 03_source_labeling_functions.ipynb")

02 COMPLETE -> /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/AeroPulse_ML_Notebooks/source_likelihood/data/source/processed/features/source_evidence_features.parquet (258 columns)
Next: 03_source_labeling_functions.ipynb


## Findings

**8 of 12 requested feature families already existed.** Upwind fire (section
15), PM10 and the CAMS dust indicator (section 16), upwind PM2.5 and the
wind vector (section 17), SO2/NO2/CO (section 19) and the geography columns
(section 20) are all upstream. Four were genuinely missing and are built
here: pollutant ratios, pollutant temporal signatures, transport consistency
and station context.

**The plan's section 13 ratios do not work as evidence, and the measurement
says so.** Median ratio by concentration band:

| band | NO2:PM2.5 | CO:PM2.5 | SO2:PM2.5 | PM10:PM2.5 |
|---|---|---|---|---|
| <30 | 0.488 | 14.40 | 0.455 | 2.069 |
| 30-60 | 0.243 | 8.13 | 0.261 | 1.175 |
| 60-120 | 0.199 | 6.29 | 0.200 | 0.847 |
| 120-250 | 0.143 | 4.37 | 0.123 | 0.557 |
| >250 | 0.071 | 1.72 | 0.059 | 0.305 |

Every ratio falls monotonically as PM2.5 rises, by a factor of 5-8. That is
arithmetic, not chemistry: PM2.5 is the denominator. Taken literally, a
labeling function reading "high NO2:PM2.5 means traffic" would fire
preferentially **on clean air** — the opposite of what a traffic episode
looks like.

Per-station normalisation was the obvious fix and it **did not work**.
Dividing each ratio by the station's own median removes the between-station
level difference but not the denominator effect; the enrichment still decayed
from 1.47 to 0.23 across the same bands. Recorded because it is the kind of
correction that looks convincing and is not.

**What works is the pollutant level anomaly**, which has no PM2.5 in it at
all — each pollutant against its own station-median level:

| band | NO2 | CO | SO2 | O3 | coarse fraction |
|---|---|---|---|---|---|
| <30 | 0.761 | 0.723 | 0.727 | 0.903 | 1.546 |
| 60-120 | 1.400 | 1.476 | 1.361 | 1.031 | 0.633 |
| >250 | 1.661 | 1.467 | 1.295 | 0.779 | 0.228 |

Now the signal points the right way and reads as physics: combustion
pollutants rise together with PM2.5 (NO2 0.76 → 1.66), ozone falls (0.90 →
0.78, consistent with NOx titration under high-emission stagnant
conditions), and the **coarse fraction collapses from 1.55 to 0.23**.

That last row is a substantive finding for this whole pipeline: severe
PM2.5 episodes in this dataset are fine-dominated, i.e. combustion, not
dust. Any dust hypothesis should therefore expect to fire on *moderate*
PM2.5 with a high coarse fraction, not on the worst hours. All three ratio
labeling functions were repointed at level anomalies as a result.

**Transport consistency is usable.** Median 6 h directional consistency is
0.964 and 68.5% of rows have a steady direction (>0.9), so the feature
discriminates rather than saturating — it identifies the third of hours
where the wind is *not* steady, which is where a transport claim should be
weakened.

**Station context separates, weakly.** The behavioural proxy assigns 57
background-like, 38 traffic-like, 34 industrial-like and 20
residential-like stations. The separations are real but small:
traffic-like stations have a morning ratio of 0.987 against 0.745-0.797
elsewhere, and industrial-like have a night ratio of 1.152 against ~1.01.
A ~25% contrast in the morning ratio is a usable feature and nothing like a
station classification; it is labelled a proxy for that reason. The
tampering test confirms it is fitted on train only — multiplying every
post-cutoff PM2.5 by 20 leaves it bit-identical.